In [ ]:
import os, json, subprocess, sys, time, glob, shutil, urllib.request
print('python', sys.version.split()[0])
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:1200])
gpu = subprocess.run(['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().splitlines()
total_mem = sum(int(line.split()[0].replace(',', '')) for line in gpu if line)
print('GPU count:', len(gpu), '| total VRAM MiB:', total_mem)


In [ ]:
os.chdir('/kaggle/working')
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mattdani21/ModelSwapper.git'], capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-300:])
os.chdir('/kaggle/working/ModelSwapper')
print(subprocess.run(['git', 'log', '-1', '--format=%h %ci'], capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pytest'], capture_output=True, text=True)
print('pytest install rc:', r.returncode)


In [ ]:
os.chdir('/kaggle/working')
SERVER = '/kaggle/working/llama-server'
if not os.path.exists(SERVER):
    rel = json.load(urllib.request.urlopen('https://api.github.com/repos/ggml-org/llama.cpp/releases/latest'))
    asset = next(a for a in rel['assets'] if 'bin-ubuntu-x64' in a['name'])
    print('release:', rel['tag_name'], '| asset:', asset['name'])
    subprocess.run(['wget', '-q', '-O', '/kaggle/working/llama.zip', asset['browser_download_url']], check=True, timeout=1200)
    subprocess.run(['unzip', '-o', '-q', '/kaggle/working/llama.zip', '-d', '/kaggle/working/llama-rel'], check=True, timeout=300)
    found = glob.glob('/kaggle/working/llama-rel/**/llama-server', recursive=True)
    print('found:', found)
    shutil.copy(found[0], SERVER)
    os.chmod(SERVER, 0o755)
print('llama-server exists:', os.path.exists(SERVER))
os.environ['LLAMA_SERVER'] = SERVER
os.environ['PATH'] = os.path.dirname(SERVER) + ':' + os.environ['PATH']


In [ ]:
INPUT = '/kaggle/input/swapos-ggufs'
MODEL_PATHS = {
    'reason': INPUT + '/Qwen3-14B-Q4_K_M.gguf',
    'code': INPUT + '/Qwen3-Coder-30B-A3B-Instruct-Q3_K_M.gguf',
    'review': INPUT + '/Qwen3-14B-Q4_K_M.gguf',
}
for p in MODEL_PATHS.values():
    ok = os.path.exists(p)
    print(p, '->', ok, round(os.path.getsize(p) / 1e9, 2), 'GB' if ok else '')
    assert ok, 'model file missing: ' + p


In [ ]:
os.chdir('/kaggle/working/ModelSwapper')
env = dict(os.environ)
env['LLAMA_CONTEXT'] = '4096'
env['LLAMA_NGPU'] = '99'
cmd = [sys.executable, 'pipeline/run_pipeline.py',
       '--models-json', json.dumps(MODEL_PATHS),
       '--out', '/kaggle/working/pipeline-results.json',
       '--capsule-dir', '/kaggle/working/capsules',
       '--port-base', '8950',
       '--max-iterations', '3',
       '--max-tokens', '2048']
print('running pipeline eval...')
t0 = time.time()
r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=8 * 3600)
print('pipeline rc:', r.returncode, '| wall:', round((time.time() - t0) / 60, 1), 'min')
print((r.stdout or '')[-2500:])
print((r.stderr or '')[-1000:])


In [ ]:
os.chdir('/kaggle/working')
res_path = '/kaggle/working/pipeline-results.json'
if os.path.exists(res_path):
    d = json.load(open(res_path))
    print('PASS RATE:', d.get('pass_rate'), '|', d.get('tasks_passed'), '/', d.get('tasks_total'))
    print('per_category:', d.get('per_category'))
    print('mean_wall_clock_s:', d.get('mean_wall_clock_s'))
    print('mean_load_s:', d.get('mean_load_s'), '| mean_evict_s:', d.get('mean_evict_s'))
    with open('/kaggle/working/pipeline-summary.txt', 'w') as f:
        f.write(json.dumps({k: v for k, v in d.items() if k != 'results'}, indent=2))
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working', 'pipeline-results.json')
shutil.make_archive('/kaggle/working/capsules', 'zip', '/kaggle/working/capsules')
print('outputs:', sorted(glob.glob('/kaggle/working/results*') + glob.glob('/kaggle/working/capsules*')))
